# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaikhabdullahwaseem17-byte/vigilant-meme/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My Rule: An item is considered 'high priority' if it has been recently viewed frequently and has a high engagement rate, indicating current user interest and interaction.

Reason Codes:
- frequent_views: Item has received a high number of views recently.
- high_engagement: Users are actively interacting with the item (e.g., clicks, time spent).
- currently_popular: Combination of frequent views and high engagement, indicating a trending item.

In [3]:
with open('/content/README (5).md', 'r') as f:
    readme_content = f.read()
print(readme_content)

# Now that we have the README content, let's proceed to Box 1 as per the instructions.

# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write +

In [4]:
with open('/content/SKILL (5).md', 'r') as f:
    skill_content = f.read()
print(skill_content)

---
name: building-baselines
description: Builds the transparent rule-based baseline every model must beat — a hand-written score with reason codes, ranked output, and precision@K evaluation. Use before training any model, or when someone reports model results with nothing to compare against.
---

# Building baselines

A model without a baseline is a number without a meaning. The baseline is a rule a human can
read — and its job is to be honestly beatable.

## Build it in this order

**1. Say the rule in plain words first.** "A page is worth reviewing if it used to get traffic,
it's getting old, and its position is slipping." If you can't say it, you can't code it.

**2. Code it as a transparent score.** Multiply/add simple conditions; no fitted weights:

```python
stale   = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions"]     # readable on purpose
```

**3. Attach reason codes.** Every score

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
import pandas as pd
import numpy as np
import os

# Create dummy data for demonstration
np.random.seed(42)
num_items = 100
data = {
    'item_id': range(num_items),
    'views': np.random.randint(100, 5000, num_items),
    'engagement_rate': np.random.rand(num_items),
    'label': np.random.randint(0, 2, num_items) # Dummy label for precision@K
}
df = pd.DataFrame(data)

print("Sample DataFrame created:")
display(df.head())

Sample DataFrame created:


,item_id,views,engagement_rate,label
0,0,960,0.729606,1
1,1,3872,0.637557,1
2,2,3192,0.887213,0
3,3,566,0.472215,1
4,4,4526,0.119594,1


Here, I'm creating a sample DataFrame with `item_id`, `views`, `engagement_rate`, and a dummy `label` column. The `views` and `engagement_rate` will be used for our scoring. You would replace this with your actual data loading and preprocessing.

In [6]:
# Define thresholds for the rule components
VIEW_THRESHOLD = 2500  # Example: frequent views if > 2500
ENGAGEMENT_THRESHOLD = 0.6  # Example: high engagement if > 0.6

# Apply the rule components
df['is_frequent_views'] = (df['views'] > VIEW_THRESHOLD).astype(int)
df['is_high_engagement'] = (df['engagement_rate'] > ENGAGEMENT_THRESHOLD).astype(int)

# Code the score: A simple additive/multiplicative score
# The rule is: 'high priority' if it has been recently viewed frequently AND has a high engagement rate
df['score'] = df['is_frequent_views'] * df['is_high_engagement'] * (df['views'] * df['engagement_rate'])

# Attach reason codes
def assign_reason_code(row):
    if row['is_frequent_views'] and row['is_high_engagement']:
        return 'currently_popular'
    elif row['is_frequent_views']:
        return 'frequent_views'
    elif row['is_high_engagement']:
        return 'high_engagement'
    else:
        return 'low_priority'

df['reason_code'] = df.apply(assign_reason_code, axis=1)

print("DataFrame with scores and reason codes:")
display(df.head())

DataFrame with scores and reason codes:


,item_id,views,engagement_rate,label,is_frequent_views,is_high_engagement,score,reason_code
0,0,960,0.729606,1,0,1,0.000000,high_engagement
1,1,3872,0.637557,1,1,1,2468.622529,currently_popular
2,2,3192,0.887213,0,1,1,2831.983074,currently_popular
3,3,566,0.472215,1,0,0,0.000000,low_priority
4,4,4526,0.119594,1,1,0,0.000000,frequent_views


This code block implements the scoring logic and assigns reason codes based on the rule defined in Box 1. I've set example thresholds for `views` and `engagement_rate`. The `score` is calculated as a product, making it zero if either condition is not met, and scales with the magnitude of views and engagement when both are true. The `reason_code` function categorizes each item based on which conditions it satisfies.

In [7]:
# Function to calculate precision@K (from SKILL (5).md)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Rank the DataFrame by score in descending order
df_ranked = df.sort_values(by='score', ascending=False).reset_index(drop=True)

# Evaluate precision@K
k_value = 20 # Evaluate for the top 20 items
actual_labels = df_ranked['label'] # Using the dummy label for demonstration

prec_at_k = precision_at_k(df_ranked['score'], actual_labels, k_value)
base_rate = actual_labels.mean()

print(f"Precision@{k_value}: {prec_at_k:.2f}")
print(f"Base Rate (overall positive labels): {base_rate:.2f}")

print("Ranked DataFrame preview:")
display(df_ranked.head(k_value))

Precision@20: 0.60
Base Rate (overall positive labels): 0.57
Ranked DataFrame preview:


,item_id,views,engagement_rate,label,is_frequent_views,is_high_engagement,score,reason_code
0,68,4651,0.985650,0,1,1,4584.260262,currently_popular
1,96,4698,0.924694,0,1,1,4344.210619,currently_popular
2,29,4758,0.871461,1,1,1,4146.409488,currently_popular
3,64,4397,0.908266,1,1,1,3993.645101,currently_popular
4,35,3990,0.896091,1,1,1,3575.404287,currently_popular
5,75,4987,0.632306,1,1,1,3153.309177,currently_popular
6,92,3199,0.936730,1,1,1,2996.599234,currently_popular
7,88,4614,0.645173,1,1,1,2976.827255,currently_popular
8,34,3656,0.807440,1,1,1,2952.001207,currently_popular
9,2,3192,0.887213,0,1,1,2831.983074,currently_popular


Here, I've used the `precision_at_k` function from the skill file to evaluate the effectiveness of our rule for the top `k_value` items. The base rate is also provided for context. The DataFrame is then sorted by the calculated `score`.

In [8]:
# Create the output directory if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# Write the ranked output to a CSV file
output_path = 'work/outputs/baseline_action_score.csv'
df_ranked.to_csv(output_path, index=False)

print(f"Ranked output saved to {output_path}")

Ranked output saved to work/outputs/baseline_action_score.csv


The ranked DataFrame, including the scores and reason codes, has been saved to `work/outputs/baseline_action_score.csv` as required by the assignment.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


For each of the top 20 items identified by your rule, fill in the following table. This is a critical step to understand your rule's behavior and identify potential issues.

| Item ID | Action (e.g., 'Review') | Reason Code (from your rule) | Confidence Note (e.g., 'High', 'Medium', 'Low') | What would make it wrong? (e.g., 'Low actual user engagement') |
|---------|-------------------------|------------------------------|-------------------------------------------------|----------------------------------------------------------------|
| 68      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 96      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 29      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 64      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 35      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 75      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 92      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 88      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 34      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 2       | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 18      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 76      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 90      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 5       | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 48      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 6       | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 1       | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 60      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 27      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 97      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


For each of the top 20 items identified by your rule, fill in the following table. This is a critical step to understand your rule's behavior and identify potential issues.

| Item ID | Action (e.g., 'Review') | Reason Code (from your rule) | Confidence Note (e.g., 'High', 'Medium', 'Low') | What would make it wrong? (e.g., 'Low actual user engagement') |
|---------|-------------------------|------------------------------|-------------------------------------------------|----------------------------------------------------------------|
| 68      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 96      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 29      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 64      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 35      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 75      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 92      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 88      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 34      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 2       | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 18      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 76      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 90      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 5       | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 48      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 6       | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 1       | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |
| 60      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 27      | Review                  | currently_popular            | Medium                                          | Incorrectly identified as popular (label=0)                    |
| 97      | Review                  | currently_popular            | Medium                                          | None (Correctly identified)                                    |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


After reviewing the top 20 items, consider the following questions to check for weak picks and potential data leakage:

*   **Which picks, if any, stood out as wrong or unexpected given your rule?**
    *   Items such as `item_id` 68, 96, 2, 18, 76, 5, 60, and 27 stood out as 'wrong picks'. These items were ranked highly by the rule ('currently_popular' reason code) but had a `label` of 0 in our dummy dataset, indicating they were not a 'positive' outcome according to the target variable.

*   **Why do you think these picks were identified by the rule but seem incorrect?**
    *   These items were identified by the rule because they met the `VIEW_THRESHOLD` and `ENGAGEMENT_THRESHOLD` (e.g., high views and high engagement rate). The rule is designed to flag items with these characteristics. However, in our dummy data, even with high views and engagement, the `label` was sometimes 0. This suggests that the simple rule, while identifying activity, doesn't perfectly correlate with the true 'positive' label in all cases, leading to false positives.

*   **Did you notice any patterns in the 'wrong' picks that suggest an issue with the rule's logic or data inputs?**
    *   The pattern in 'wrong' picks is that they consistently met both criteria (`is_frequent_views` and `is_high_engagement`) resulting in a 'currently_popular' reason code and a high score, but their `label` was 0. This indicates that while the rule correctly identifies items with high views and engagement, these two features alone (and their multiplicative combination) are not always sufficient to predict a 'positive' outcome as defined by the `label`. It suggests that either the thresholds could be refined, or additional features might be needed to better capture the 'true' positive cases.

*   **Confirm that no product flags or future windows leaked into your rule. For example, is your rule inadvertently using data that would not have been available at the time of prediction, or is it directly using a 'true label' that represents the outcome you are trying to predict (which would be leakage)?**
    *   In this baseline rule, we explicitly used `views` and `engagement_rate` as features that would be available *before* a prediction is made. The `label` column was used *only* for evaluating `precision@K` and identifying weak picks, not as an input to the scoring rule itself. Therefore, there is no direct leakage of the 'true label' into the rule's scoring mechanism. The features (`views`, `engagement_rate`) are assumed to represent past behavior, not future outcomes or specific product flags that reveal the outcome prematurely.

*   **Does the data used for `views` or `engagement_rate` unintentionally include any information about the *outcome* itself, rather than just the *features* that predict the outcome?**
    *   Assuming `views` and `engagement_rate` are recorded *before* the 'outcome' represented by the `label` is determined, they should be valid predictive features. If, however, 'engagement_rate' was, for instance, calculated based on user interactions *after* the 'outcome' was already known (or influenced by it), that could be a form of leakage. For this dummy data, we assume these metrics are pure features available at prediction time. In a real-world scenario, careful definition and timing of feature collection would be crucial to avoid such implicit leakage.

After reviewing the top 20 items, consider the following questions to check for weak picks and potential data leakage:

*   Which picks, if any, stood out as wrong or unexpected given your rule?
*   Why do you think these picks were identified by the rule but seem incorrect?
*   Did you notice any patterns in the 'wrong' picks that suggest an issue with the rule's logic or data inputs?
*   Confirm that no product flags or future windows leaked into your rule. For example, is your rule inadvertently using data that would not have been available at the time of prediction, or is it directly using a 'true label' that represents the outcome you are trying to predict (which would be leakage)?
*   Does the data used for `views` or `engagement_rate` unintentionally include any information about the *outcome* itself, rather than just the *features* that predict the outcome?

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.